## Build and test the container 

This notebook builds and tests the container with the Python command line tool.

Link to doc: https://eoap.github.io/quickwin/containers/water_bodies/

Recreate the app.py file as it going to be copied to the container:

In [ ]:
(
set -euo pipefail
cat ../water-bodies/command-line-tools/detect-water-body/detect_water_body/main.py
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Recreate the Dockerfile:

In [ ]:
(
set -euo pipefail
package=../water-bodies/command-line-tools/detect-water-body
for resource in Dockerfile .dockerignore pyproject.toml app.py README.md LICENSE.md requirements detect_water_body; do
    cp -R "$package/$resource" .
done
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Use podman to build the container:

In [ ]:
(
set -euo pipefail
podman build --format docker -t localhost/detect-water-body:latest .
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


List the container images and get the one just built:

In [ ]:
(
set -euo pipefail
podman images | grep "detect-water-body"
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Print the Python command line tool help (it's part of the test):

In [ ]:
(
set -euo pipefail
podman run --rm -it --env=PYTHONPATH=/app localhost/detect-water-body python -m app --help
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Now the final test, run the Python command line tool with all the arguments:

In [ ]:
(
set -euo pipefail
mkdir -p ${QUICKWIN_WORKSPACE:-/workspace/quickwin}/runs
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


In [ ]:
(
set -euo pipefail
podman run \
    -i \
    --userns=keep-id \
    --mount=type=bind,source=${QUICKWIN_WORKSPACE:-/workspace/quickwin}/runs,target=/runs \
    --workdir=/runs \
    --read-only=true \
    --user=$(id -u):$(id -g) \
    --rm \
    --env=HOME=/runs \
    --env=PYTHONPATH=/app \
    localhost/detect-water-body:latest \
    python \
    -m \
    app \
    --aoi \
    "-121.399,39.834,-120.74,40.472" \
    --band \
    green \
    --band \
    nir \
    --epsg \
    "EPSG:4326" \
    --input-item \
    https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A

)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Use stac tools to inspect the generated STAC Catalog:

In [ ]:
(
set -euo pipefail
stac describe ${QUICKWIN_WORKSPACE:-/workspace/quickwin}/runs/catalog.json
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi
